# Retrieval-Augmented Generation (RAG) & Vector Search

This notebook teaches you how RAG works from first principles. By the end, you will understand how to chunk documents, embed them into vectors, perform similarity search, and assemble a RAG prompt — all with lightweight Python libraries and no external vector database required.

## 1. What is RAG?

Large Language Models are trained on a fixed snapshot of data. Once training ends, the model has no awareness of events, documents, or knowledge that appeared after its cutoff date. Even within its training window, the model may not have seen your proprietary documents, internal wikis, or domain-specific data. This means that out of the box, an LLM can confidently produce answers that are outdated, incomplete, or simply fabricated.

Retrieval-Augmented Generation (RAG) addresses this limitation by adding a **retrieval step** before generation. Instead of asking the model to answer purely from its parameters, we first search a knowledge base for documents that are relevant to the user's question. Those documents are then injected directly into the prompt as context, giving the model grounded, up-to-date information to reason over.

The key insight is that we separate **what the model knows** (its parametric memory) from **what we supply at inference time** (the retrieved context). This means we can update our knowledge base at any time — adding new documents, correcting outdated ones, or removing irrelevant material — without retraining the model. RAG turns a static LLM into a dynamic system that can answer questions about any corpus we point it at.

RAG has become the standard architecture for question-answering systems, enterprise chatbots, and code assistants. It dramatically reduces hallucination, improves factual accuracy, and provides a natural way to cite sources — since every answer can be traced back to the specific documents that were retrieved.

## 2. The RAG Pipeline

A typical RAG system follows four steps:

### Step 1 — Chunk Documents
Raw documents (PDFs, web pages, text files) are split into smaller pieces called **chunks**. Chunking is necessary because embedding models have token limits, and smaller passages produce more focused, meaningful embeddings. Common strategies include splitting by sentence, by fixed character windows with overlap, or by semantic boundaries.

### Step 2 — Embed Chunks into Vectors
Each chunk is passed through an **embedding model** that maps text to a dense numerical vector (typically 384 to 1536 dimensions). Texts with similar meaning end up close together in this vector space, even if they use different words.

### Step 3 — Store in a Vector Database
The resulting vectors (along with metadata and the original text) are stored in an index optimized for similarity search. This can be a dedicated vector database like FAISS, Oracle AI Vector Search, or Chroma, or simply a NumPy array for small-scale experiments like this notebook.

### Step 4 — Retrieve and Generate
At query time, the user's question is embedded with the same model. We compute similarity (usually cosine similarity) between the query vector and all stored chunk vectors, retrieve the **top-k** most similar chunks, and inject them into the LLM prompt as context. The model then generates an answer grounded in those retrieved passages.

```
User Query
    │
    ▼
┌──────────┐     ┌─────────────────┐     ┌───────────┐
│  Embed   │────▶│  Vector Search   │────▶│  Top-K    │
│  Query   │     │  (cosine sim)    │     │  Chunks   │
└──────────┘     └─────────────────┘     └─────┬─────┘
                                               │
                                               ▼
                                    ┌──────────────────┐
                                    │  Assemble Prompt  │
                                    │  Context + Query  │
                                    └────────┬─────────┘
                                             │
                                             ▼
                                    ┌──────────────────┐
                                    │    LLM Answer     │
                                    └──────────────────┘
```

---
## 3. Setup

We use three lightweight libraries so this notebook runs anywhere:
- **sentence-transformers** — pre-trained embedding models
- **numpy** — array operations
- **scikit-learn** — cosine similarity and dimensionality reduction

In [ ]:
# Install dependencies if needed (uncomment the line below)
# !pip install sentence-transformers numpy scikit-learn matplotlib

from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import textwrap

print("All imports successful.")

---
## 4. Sample Documents

We create a small corpus of educational passages covering core AI and ML topics. Each document is a short paragraph — the kind of content you might extract from a textbook chapter or documentation page.

In [ ]:
documents = [
    {
        "id": "ai_history",
        "text": (
            "The field of Artificial Intelligence was formally founded at the Dartmouth Conference in 1956, "
            "where John McCarthy, Marvin Minsky, Nathaniel Rochester, and Claude Shannon proposed that "
            "every aspect of learning could in principle be so precisely described that a machine could be "
            "made to simulate it. Early AI research focused on symbolic reasoning, theorem proving, and "
            "game playing. The field went through several cycles of enthusiasm and disappointment known as "
            "AI winters, before the modern resurgence driven by deep learning and massive datasets."
        ),
    },
    {
        "id": "transformer_arch",
        "text": (
            "The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need' by "
            "Vaswani et al. It replaced recurrent layers with self-attention mechanisms, allowing the model "
            "to process all tokens in a sequence in parallel rather than sequentially. A Transformer consists "
            "of an encoder and a decoder, each built from stacked layers of multi-head self-attention and "
            "feed-forward networks. The self-attention mechanism computes a weighted sum of all positions in "
            "the sequence, where the weights are determined by the compatibility between query and key vectors. "
            "Positional encodings are added to give the model information about token order."
        ),
    },
    {
        "id": "cnn_basics",
        "text": (
            "Convolutional Neural Networks (CNNs) are a class of deep neural networks most commonly applied "
            "to image recognition tasks. A CNN applies learnable filters (kernels) that slide across the input "
            "image to produce feature maps. Pooling layers reduce the spatial dimensions while retaining the "
            "most important features. Landmark CNN architectures include LeNet-5 for digit recognition, "
            "AlexNet which won the 2012 ImageNet competition, VGGNet known for its uniform architecture, "
            "and ResNet which introduced skip connections to train very deep networks."
        ),
    },
    {
        "id": "reinforcement_learning",
        "text": (
            "Reinforcement Learning (RL) is a paradigm where an agent learns to make decisions by interacting "
            "with an environment. The agent observes states, takes actions, and receives rewards or penalties. "
            "The goal is to learn a policy that maximizes the cumulative reward over time. Key algorithms include "
            "Q-learning, which estimates the value of state-action pairs, and policy gradient methods like "
            "REINFORCE and PPO. Deep RL combines neural networks with RL, achieving breakthroughs like "
            "AlphaGo defeating the world champion in Go and agents learning complex locomotion in simulation."
        ),
    },
    {
        "id": "data_preprocessing",
        "text": (
            "Data preprocessing is a critical step in any machine learning pipeline. Raw data often contains "
            "missing values, outliers, inconsistent formats, and noise that can degrade model performance. "
            "Common preprocessing steps include normalization (scaling features to a common range), "
            "standardization (zero mean, unit variance), one-hot encoding of categorical variables, and "
            "imputation of missing values. Feature engineering — creating new informative features from raw "
            "data — often has a larger impact on performance than the choice of model architecture."
        ),
    },
    {
        "id": "transfer_learning",
        "text": (
            "Transfer learning is the practice of taking a model trained on a large dataset and fine-tuning it "
            "on a smaller, task-specific dataset. This is effective because early layers of neural networks learn "
            "general features (edges, textures, basic patterns) that are useful across many tasks. In NLP, "
            "models like BERT and GPT are pre-trained on massive text corpora and then fine-tuned for specific "
            "tasks such as sentiment analysis, named entity recognition, or question answering. Transfer "
            "learning dramatically reduces the amount of labeled data and compute required to achieve "
            "strong performance on downstream tasks."
        ),
    },
    {
        "id": "embeddings",
        "text": (
            "Word embeddings map words to dense, low-dimensional vectors where semantic relationships are "
            "preserved. Word2Vec, introduced by Mikolov et al. in 2013, showed that vector arithmetic could "
            "capture analogies: king - man + woman ≈ queen. GloVe (Global Vectors) improved on this by "
            "incorporating global co-occurrence statistics. Modern sentence and document embeddings extend "
            "this idea to longer text spans. Models like Sentence-BERT produce fixed-size vectors for entire "
            "sentences, enabling efficient semantic search and clustering over large text collections."
        ),
    },
    {
        "id": "attention_mechanism",
        "text": (
            "The attention mechanism allows a model to focus on the most relevant parts of the input when "
            "producing each element of the output. In its simplest form, attention computes a weighted "
            "average of value vectors, where the weights are derived from the similarity between a query "
            "vector and a set of key vectors. Scaled dot-product attention divides the dot products by the "
            "square root of the key dimension to prevent extremely large values. Multi-head attention runs "
            "several attention operations in parallel, each with different learned projections, and "
            "concatenates their outputs. This allows the model to jointly attend to information from "
            "different representation subspaces at different positions."
        ),
    },
    {
        "id": "llm_training",
        "text": (
            "Large Language Models are trained in two main phases. In pre-training, the model learns to "
            "predict the next token on trillions of tokens from books, web pages, and code. This phase "
            "is extremely compute-intensive — GPT-4 class models require thousands of GPUs running for "
            "months. The second phase is alignment, where the model is fine-tuned using supervised "
            "instruction-following examples and reinforcement learning from human feedback (RLHF). "
            "Alignment teaches the model to follow instructions, refuse harmful requests, and produce "
            "helpful, coherent responses rather than simply completing text."
        ),
    },
    {
        "id": "vector_databases",
        "text": (
            "Vector databases are purpose-built storage systems optimized for storing, indexing, and "
            "querying high-dimensional vectors. Unlike traditional databases that use B-tree or hash "
            "indexes, vector databases use approximate nearest neighbor (ANN) algorithms such as HNSW "
            "(Hierarchical Navigable Small World) and IVF (Inverted File Index) to find similar vectors "
            "efficiently. Popular options include FAISS (by Meta), Oracle AI Vector Search (integrated "
            "into Oracle Database), Chroma, Pinecone, Weaviate, and Milvus. These systems support "
            "metadata filtering, hybrid search combining vector and keyword matching, and can scale "
            "to billions of vectors."
        ),
    },
]

print(f"Loaded {len(documents)} documents.")
for doc in documents:
    print(f"  - {doc['id']:25s} ({len(doc['text'])} chars)")

---
## 5. Chunking

Documents are often too long to embed as a single unit. We split them into smaller, overlapping chunks so that each chunk captures a coherent idea and the overlap prevents losing information at boundaries.

Below we implement two common strategies:
1. **Sentence-based chunking** — split on sentence boundaries
2. **Fixed-window chunking** — split by character count with configurable overlap

In [ ]:
import re


def chunk_by_sentences(text: str, max_sentences: int = 2) -> list[str]:
    """Split text into chunks of `max_sentences` sentences each.
    Consecutive chunks overlap by one sentence for continuity."""
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    chunks = []
    step = max(1, max_sentences - 1)  # overlap of 1 sentence
    for i in range(0, len(sentences), step):
        chunk = " ".join(sentences[i : i + max_sentences])
        if chunk:
            chunks.append(chunk)
    return chunks


def chunk_by_characters(text: str, window: int = 200, overlap: int = 50) -> list[str]:
    """Split text into fixed-size character windows with overlap."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + window
        chunks.append(text[start:end].strip())
        start += window - overlap
    return chunks


# --- Apply sentence-based chunking to all documents ---
all_chunks = []       # list of chunk text strings
chunk_sources = []    # parallel list tracking which document each chunk came from

for doc in documents:
    doc_chunks = chunk_by_sentences(doc["text"], max_sentences=2)
    for chunk in doc_chunks:
        all_chunks.append(chunk)
        chunk_sources.append(doc["id"])

print(f"Total chunks: {len(all_chunks)} (from {len(documents)} documents)\n")

# Show a few example chunks
for i, (chunk, source) in enumerate(zip(all_chunks[:6], chunk_sources[:6])):
    wrapped = textwrap.fill(chunk, width=90)
    print(f"Chunk {i} [{source}]:\n{wrapped}\n")

Let's also demonstrate character-based chunking on a single document so you can see the difference:

In [ ]:
# Character-based chunking demo on one document
char_chunks = chunk_by_characters(documents[1]["text"], window=200, overlap=50)

print(f"Character-based chunks for '{documents[1]['id']}' (window=200, overlap=50):\n")
for i, chunk in enumerate(char_chunks):
    print(f"  Chunk {i} ({len(chunk)} chars): {chunk[:80]}...")

print(f"\nWe will use the sentence-based chunks for the rest of this notebook.")

---
## 6. Embedding

We use `all-MiniLM-L6-v2`, a lightweight but effective sentence embedding model. It maps each chunk to a **384-dimensional** vector. The model is small enough to run on a CPU in seconds.

In [ ]:
# Load the embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed all chunks
chunk_embeddings = model.encode(all_chunks, show_progress_bar=True)

print(f"\nEmbedding matrix shape: {chunk_embeddings.shape}")
print(f"  - {chunk_embeddings.shape[0]} chunks")
print(f"  - {chunk_embeddings.shape[1]} dimensions per embedding")
print(f"  - dtype: {chunk_embeddings.dtype}")
print(f"  - Total memory: {chunk_embeddings.nbytes / 1024:.1f} KB")

---
## 7. Visualization 1 — Cosine Similarity Heatmap

We compute the pairwise cosine similarity between all chunk embeddings and display it as a heatmap. Chunks from the same document (or related topics) should show higher similarity values.

In [ ]:
# Compute pairwise cosine similarity
sim_matrix = cosine_similarity(chunk_embeddings)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(sim_matrix, cmap="YlOrRd", vmin=0, vmax=1)

# Label ticks with source document IDs
tick_labels = [f"{i}:{src[:12]}" for i, src in enumerate(chunk_sources)]
ax.set_xticks(range(len(tick_labels)))
ax.set_xticklabels(tick_labels, rotation=90, fontsize=7)
ax.set_yticks(range(len(tick_labels)))
ax.set_yticklabels(tick_labels, fontsize=7)

ax.set_title("Cosine Similarity Between All Chunk Embeddings", fontsize=14)
plt.colorbar(im, ax=ax, label="Cosine Similarity")
plt.tight_layout()
plt.show()

print("\nNotice the bright diagonal blocks — chunks from the same document are more similar.")
print("Off-diagonal bright spots reveal cross-document topical similarity")
print("(e.g., 'transformer_arch' chunks are similar to 'attention_mechanism' chunks).")

---
## 8. Visualization 2 — 2D Embedding Projection

We project the 384-dimensional embeddings down to 2D using PCA (for a quick global view) and t-SNE (for better local structure). Points are colored by source document to reveal clustering.

In [ ]:
# Assign a color to each unique source document
unique_sources = list(dict.fromkeys(chunk_sources))  # preserve order
cmap = plt.cm.get_cmap("tab10", len(unique_sources))
color_map = {src: cmap(i) for i, src in enumerate(unique_sources)}
colors = [color_map[src] for src in chunk_sources]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- PCA ---
pca = PCA(n_components=2, random_state=42)
coords_pca = pca.fit_transform(chunk_embeddings)

ax = axes[0]
for src in unique_sources:
    mask = [s == src for s in chunk_sources]
    pts = coords_pca[mask]
    ax.scatter(pts[:, 0], pts[:, 1], c=[color_map[src]], label=src, s=60, alpha=0.8, edgecolors="k", linewidths=0.5)
ax.set_title("PCA Projection of Chunk Embeddings", fontsize=13)
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
ax.legend(fontsize=7, loc="best", ncol=2)
ax.grid(True, alpha=0.3)

# --- t-SNE ---
# Adjust perplexity for small datasets (must be < n_samples)
perplexity = min(10, len(all_chunks) - 1)
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity, n_iter=1000)
coords_tsne = tsne.fit_transform(chunk_embeddings)

ax = axes[1]
for src in unique_sources:
    mask = [s == src for s in chunk_sources]
    pts = coords_tsne[mask]
    ax.scatter(pts[:, 0], pts[:, 1], c=[color_map[src]], label=src, s=60, alpha=0.8, edgecolors="k", linewidths=0.5)
ax.set_title("t-SNE Projection of Chunk Embeddings", fontsize=13)
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(fontsize=7, loc="best", ncol=2)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Chunks from the same document cluster together.")
print("Related topics (e.g., transformers + attention) appear nearby in the embedding space.")

---
## 9. Vector Search

Now we build the core retrieval function. Given a natural-language query, we:
1. Embed the query with the same model
2. Compute cosine similarity against all chunk embeddings
3. Return the top-k most similar chunks

In [ ]:
def search(query: str, top_k: int = 3) -> list[dict]:
    """Embed the query and return the top-k most similar chunks."""
    query_embedding = model.encode([query])  # shape: (1, 384)
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]  # shape: (n_chunks,)

    # Get indices of top-k highest similarities
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "rank": len(results) + 1,
            "chunk_id": int(idx),
            "source": chunk_sources[idx],
            "similarity": float(similarities[idx]),
            "text": all_chunks[idx],
        })
    return results


def display_results(query: str, results: list[dict]):
    """Pretty-print search results."""
    print(f"{'='*90}")
    print(f"QUERY: {query}")
    print(f"{'='*90}")
    for r in results:
        wrapped = textwrap.fill(r['text'], width=85, initial_indent="    ", subsequent_indent="    ")
        print(f"\n  [{r['rank']}] Score: {r['similarity']:.4f}  |  Source: {r['source']}  |  Chunk #{r['chunk_id']}")
        print(wrapped)
    print()

In [ ]:
# --- Demo Query 1: About attention ---
results = search("How does the attention mechanism work in transformers?")
display_results("How does the attention mechanism work in transformers?", results)

In [ ]:
# --- Demo Query 2: About image recognition ---
results = search("What neural network architecture is best for image classification?")
display_results("What neural network architecture is best for image classification?", results)

In [ ]:
# --- Demo Query 3: About training data and preparation ---
results = search("How should I clean and prepare data before training a model?")
display_results("How should I clean and prepare data before training a model?", results)

---
## 10. RAG Prompt Assembly

The final step in a RAG pipeline is formatting the retrieved chunks into a prompt for the LLM. Below is a simple but effective template. In production, you would send this prompt to an LLM API (OpenAI, Ollama, vLLM, etc.) to generate the answer.

In [ ]:
def build_rag_prompt(query: str, top_k: int = 3) -> str:
    """Retrieve relevant chunks and assemble them into a RAG prompt."""
    results = search(query, top_k=top_k)

    # Format the context section
    context_parts = []
    for r in results:
        context_parts.append(f"[Source: {r['source']}]\n{r['text']}")
    context = "\n\n".join(context_parts)

    prompt = f"""Context:
{context}

Question: {query}
Answer based on the context above:"""

    return prompt


# --- Demonstrate prompt assembly ---
query = "What is the difference between pre-training and fine-tuning a language model?"
prompt = build_rag_prompt(query, top_k=3)

print("Generated RAG Prompt:")
print("=" * 90)
print(prompt)
print("=" * 90)
print(f"\nPrompt length: {len(prompt)} characters (~{len(prompt.split())} words)")
print("\nThis prompt would be sent to an LLM to generate a grounded answer.")

In [ ]:
# A second example with a different query
query2 = "How do vector databases enable fast similarity search?"
prompt2 = build_rag_prompt(query2, top_k=3)

print("Generated RAG Prompt:")
print("=" * 90)
print(prompt2)
print("=" * 90)

---
## 11. Going Further

This notebook demonstrated a minimal but complete RAG pipeline using NumPy as a stand-in for a vector database. In a production system, you would replace the in-memory array with a dedicated solution and adopt more sophisticated chunking and retrieval strategies.

### Vector Databases
- **FAISS** (Meta) — highly optimized C++ library for ANN search; great for on-premise deployments
- **Oracle AI Vector Search** — vector indexing built directly into Oracle Database, allowing unified SQL queries over relational data and embeddings, with enterprise-grade security and scalability
- **Chroma** — Python-native, easy to prototype with; good for small-to-medium workloads
- **Pinecone / Weaviate / Milvus** — managed or self-hosted options for large-scale production

### Chunking Strategies
- **Semantic chunking** — split based on topic shifts (e.g., using embedding similarity between consecutive sentences)
- **Recursive chunking** — try splitting by paragraphs first, then sentences, then characters, to keep chunks at a target size
- **Parent-child chunking** — embed small chunks for precision, but retrieve the larger parent document for context
- **Metadata enrichment** — attach section titles, page numbers, timestamps to chunks for filtering

### Re-Ranking
- Initial vector search returns approximate candidates quickly
- A **cross-encoder re-ranker** (e.g., `cross-encoder/ms-marco-MiniLM-L-6-v2`) scores each query-chunk pair more carefully
- This two-stage pipeline (fast retrieval + precise re-ranking) is the standard in production RAG systems

### Evaluation
- **Retrieval quality**: Precision@k, Recall@k, Mean Reciprocal Rank (MRR)
- **Answer quality**: Use LLM-as-judge, RAGAS, or human evaluation to measure faithfulness and relevance
- Build evaluation datasets specific to your domain — generic benchmarks rarely reflect real-world performance